# SignalScope — ConvNeXt-Tiny + Forensic SRM Model Fine-Tuning
### Telling Real From Synthetic in the Age of Generative Media (SIH 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishvjani/sih-project/blob/master/notebooks/1_SignalScope_Model_FineTuning_Colab.ipynb)

This notebook provides end-to-end GPU-accelerated training, differential fine-tuning, Platt calibration, and explainability for **SignalScope**.

---
### 📌 Pipeline Overview
1. **Hardware & Environment Check**: Verify GPU accelerator (T4 / V100 / A100).
2. **Google Drive Mount**: Persist datasets and checkpoints.
3. **Dataset Ingestion**: Automated download of the CIFAKE benchmark dataset (100k+ labelled Real vs AI).
4. **Dual-Stream Architecture**: ConvNeXt-Tiny (Semantic Stream) + Spatial Rich Models (Forensic Residual Stream).
5. **Two-Phase Training**:
   - *Phase 1*: Linear probe with frozen backbone to adapt forensic features.
   - *Phase 2*: Differential fine-tuning on stages 3 & 4 to prevent catastrophic forgetting and overfitting.
6. **Platt Temperature Calibration**: Optimize temperature T on validation logits for honest confidence.
7. **Held-out & Unseen Generator Evaluation**: Calculate ROC-AUC, Macro-F1, and Confusion Matrix.
8. **Grad-CAM Explainability**: Visualize spatial heatmaps on unseen test images.
9. **1-Click Export**: Save trained weights directly to Google Drive.

## Step 1: Check GPU Accelerator & Environment

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Active GPU:', torch.cuda.get_device_name(0))

## Step 2: Set Up Workspace & Clone Repository
*(Google Drive is completely optional — you can train and run directly in Colab without it!)*

In [ ]:
# Google Drive is completely OPTIONAL.
# Set MOUNT_DRIVE = True only if you want to connect your Drive.
# Default is False so training starts immediately without permission popups!
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('✓ Google Drive mounted.')
    except Exception as e:
        print('ℹ Drive mount skipped:', e)
else:
    print('✓ Running directly in Colab storage (no Google Drive permissions needed).')

# Clone or update the SignalScope GitHub Repository
!git clone https://github.com/vishvjani/sih-project.git /content/sih-project || (cd /content/sih-project && git pull)
%cd /content/sih-project/repo


## Step 3: Install Required Dependencies

In [ ]:
!pip install -q -r requirements-colab.txt || pip install -q timm torchvision scikit-learn matplotlib albumentations kagglehub

## Step 4: Download CIFAKE Dataset (100k+ Images)
Using `kagglehub` to download the standard CIFAKE real-vs-synthetic dataset in 1 click.

In [ ]:
import os
import kagglehub

print('Downloading CIFAKE dataset from Kaggle...')
data_path = kagglehub.dataset_download('birdy654/cifake-real-and-ai-generated-synthetic-images')
print('Dataset downloaded to:', data_path)
os.environ['DATA_DIR'] = data_path

## Step 5: Phase 1 Training — Linear Probe (Frozen Backbone)
In Phase 1, the ConvNeXt-Tiny backbone is **FROZEN**. Only the Forensic SRM Residual Stream and the Multi-Task Classifier Heads learn. This protects ImageNet features from sudden corruption.

In [ ]:
!python -m model.train \
    --data-dir "$DATA_DIR" \
    --save-dir "./model/weights" \
    --phase1-epochs 2 \
    --phase2-epochs 0 \
    --batch-size 64 \
    --lr-head 0.001 \
    --device cuda

## Step 6: Phase 2 Fine-Tuning — Differential Stage Unfreezing
Now Stages 3 & 4 are **UNFROZEN** with a very small learning rate (`1e-5`), while the Stem and Stages 1-2 remain frozen. This allows the network to learn subtle diffusion & GAN artifacts without memorizing specific generator styles.

In [ ]:
!python -m model.train \
    --data-dir "$DATA_DIR" \
    --save-dir "./model/weights" \
    --phase1-epochs 0 \
    --phase2-epochs 5 \
    --batch-size 64 \
    --lr-head 0.0001 \
    --lr-backbone 0.00001 \
    --device cuda

## Step 7: Evaluate on Held-Out Test Set (Unseen Generator AUC)
Computes the mandatory SIH metrics: Overall ROC-AUC, Unseen-Generator Split AUC, Macro-F1, and Expected Calibration Error (ECE).

In [ ]:
!python -m model.evaluate \
    --checkpoint "./model/weights/signalscope_convnext_tiny.pth" \
    --data-dir "$DATA_DIR" \
    --batch-size 64 \
    --device cuda \
    --output "evaluation_report.json"

## Step 8: Interactive Grad-CAM & Forensic Visualizer
Test any image interactively to inspect the calibrated confidence, Grad-CAM heatmap, and generator attribution.

In [ ]:
import base64
import io
import json
import glob
import matplotlib.pyplot as plt
from PIL import Image
from model.predict import predict_image
from model.forensic import extract_visual_noise_residual

# Pick any test image from the dataset
sample_images = glob.glob(f"{data_path}/**/*.jpg", recursive=True)
if sample_images:
    test_img_path = sample_images[0]
    image = Image.open(test_img_path)
    
    result = predict_image(image, device="cuda" if torch.cuda.is_available() else "cpu")
    
    # Decode Grad-CAM heatmap overlay
    heatmap_data = base64.b64decode(result['gradcam_heatmap_base64'])
    heatmap_img = Image.open(io.BytesIO(heatmap_data))
    residual_img = extract_visual_noise_residual(image)
    
    # Plot Side-by-Side
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(image)
    axs[0].set_title('Original Image')
    axs[0].axis('off')
    
    axs[1].imshow(residual_img, cmap='gray')
    axs[1].set_title('Forensic SRM Noise Residual')
    axs[1].axis('off')
    
    axs[2].imshow(heatmap_img)
    axs[2].set_title(f"Grad-CAM: {result['verdict']} ({result['confidence_percentage']}%)")
    axs[2].axis('off')
    
    plt.suptitle(f"SignalScope Prediction: {result['verdict']} | Generator: {result['generator_attribution']['predicted_family']}", fontsize=14, fontweight='bold')
    plt.show()
    print(json.dumps(result, indent=2))

## Step 9: Export Model Weights & Configuration
Saves the fine-tuned model and triggers a direct 1-click download to your computer.

In [ ]:
import os
from google.colab import files

weights_file = './model/weights/signalscope_convnext_tiny.pth'
config_file = './model/weights/calibration_config.json'

# 1. If Google Drive is mounted, copy to Google Drive
if os.path.exists('/content/drive/MyDrive'):
    !mkdir -p /content/drive/MyDrive/SignalScope_Models
    !cp ./model/weights/signalscope_convnext_tiny.pth /content/drive/MyDrive/SignalScope_Models/ 2>/dev/null || true
    !cp ./model/weights/calibration_config.json /content/drive/MyDrive/SignalScope_Models/ 2>/dev/null || true
    !cp evaluation_report.json /content/drive/MyDrive/SignalScope_Models/ 2>/dev/null || true
    print('✓ Saved to Google Drive: MyDrive/SignalScope_Models')

# 2. Direct browser download (works with 1 click without needing Google Drive!)
print('\nDownloading fine-tuned model directly to your computer...')
try:
    files.download(weights_file)
    files.download(config_file)
    print('✓ Download triggered in your browser!')
except Exception as e:
    print(f'Weights file saved locally at: {os.path.abspath(weights_file)}')
